# Day 5: AI Company Brochure Generator

## Goal

Build a small AI application that takes a **company name + website URL** and creates a short company brochure.

We will rebuild the course project while understanding the reasoning behind every step.

### Overall flow

```text
Company URL
    ↓
Scrape landing page + links
    ↓
LLM chooses useful links
    ↓
Fetch those relevant pages
    ↓
Combine the website information
    ↓
LLM writes the brochure
    ↓
Display the brochure as Markdown
```

> **Learning rule:** Don't treat this notebook as a copy-paste exercise. Understand what data enters each function, what comes out, and why the LLM is being called.

## 1. Imports

We need three kinds of tools:

- `os` and `dotenv` for reading our API key from `.env`.
- `json` because the link-selection LLM will return structured JSON.
- `IPython.display` to render the generated brochure nicely in the notebook.
- `scraper` contains the website-fetching functions from the course project.
- `OpenAI` is the Python client we can use with Google's OpenAI-compatible Gemini endpoint.

In [ ]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

## 2. Gemini configuration

The original course notebook uses OpenAI. For our version, we use Gemini through Google's OpenAI-compatible endpoint.

The important idea is that the **client** knows where to send the request (`base_url`), while `MODEL` tells Gemini which model should process it.

Keep the API key in `.env`, never directly inside this notebook.

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")

if google_api_key and len(google_api_key) > 10:
    print("Gemini API key looks good so far")
else:
    print("There might be a problem with your Gemini API key")

MODEL = "gemini-3.5-flash-lite"

gemini = OpenAI(
    api_key=google_api_key,
    base_url=GEMINI_BASE_URL
)

## 3. First understand the raw website data

Before involving an LLM, we need to know what links exist on a website.

`fetch_website_links(url)` is our scraper function. It returns links found on the page.

At this point we are **not asking the LLM anything yet**. We are simply collecting raw information.

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

## 4. Step 1: Ask the LLM which links matter

A company website can contain many links: About, Careers, Privacy, Terms, Login, Contact, products, etc.

We don't want to manually write rules for every website. Instead, we give the list of links to the LLM and ask it to identify the useful ones.

This is a good LLM task because the model needs to understand the **meaning** of each link, not just match one fixed keyword.

We also ask for JSON so that our Python program can reliably parse the answer.

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

### Why two prompts?

We use:

- a **system prompt** for the model's job and output rules,
- a **user prompt** for the actual website-specific data.

The system prompt stays mostly the same for every company. The user prompt changes because the URL and links change.

In [ ]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company,
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

### Let's inspect the prompt before sending it

This is a useful debugging habit: first look at the exact information we are giving the model.

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

## 5. Turn the link-selection process into a function

Now we connect Python and the LLM.

The important sequence is:

1. Build the user prompt.
2. Send system + user messages to Gemini.
3. Read the model's text response.
4. Convert the JSON text into a Python dictionary using `json.loads()`.
5. Return that dictionary so the rest of our program can use it.

In [ ]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")

    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)

    print(f"Found {len(links['links'])} relevant links")
    return links

### Test it

If Gemini quota is available, this should return something like:

```text
{
  "links": [
    {"type": "about page", "url": "..."},
    {"type": "careers page", "url": "..."}
  ]
}
```

The exact links depend on the website.

In [ ]:
select_relevant_links("https://huggingface.co")

## 6. Step 2: Gather all the information needed for the brochure

Now we combine two kinds of information:

- the original landing page,
- the additional pages selected by the LLM.

This is where the project starts becoming an **agent-like multi-step workflow**: one LLM call helps decide what information the next stage should use.

In [ ]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)

    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"

    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

### What is happening inside the loop?

For every relevant link returned by Gemini, we fetch that page's contents and append it to `result`.

So the final string becomes a mini knowledge pack for the brochure-writing model.

In [ ]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## 7. Define the brochure-writing instructions

Now we need a second prompt with a different job.

The first LLM call answered: **Which pages should I read?**

The second LLM call answers: **What brochure should I write from those pages?**

Notice how the system prompt specifies the desired audience and output format.

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

### Experiment: tone is controlled by the prompt

The course also demonstrates that we can change the personality of the generated brochure simply by changing the system prompt.

For example, adding words such as `humorous`, `entertaining`, and `witty` changes the requested tone without changing the rest of the pipeline.

## 8. Build the brochure user prompt

This function puts the company name together with the website information.

The `[:5_000]` slice limits the prompt to the first 5,000 characters. This keeps the request from becoming unnecessarily large.

In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""

    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000]
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

## 9. Generate the brochure

Now we make the second LLM call.

The model receives:

- the brochure-writing instructions as the `system` message,
- the company-specific website information as the `user` message.

The model's response is plain Markdown, which we render using IPython's `Markdown()`.

In [ ]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )

    result = response.choices[0].message.content
    display(Markdown(result))

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

## 10. Final improvement: streaming

Without streaming, we wait for the complete response and then display it.

With `stream=True`, the model sends the response in chunks. We append each chunk to `response` and update the displayed Markdown.

This creates the familiar typewriter effect.

In [ ]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        update_display(
            Markdown(response),
            display_id=display_handle.display_id
        )

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

## 11. Project recap

### The important concepts from this project

1. **Web scraping** gives the application raw website information.
2. **LLM-based link selection** lets the model decide which pages are useful.
3. **JSON output** lets Python consume the LLM's structured answer.
4. **Prompt composition** lets us combine instructions with dynamically collected data.
5. **Multiple LLM calls** create a multi-step workflow rather than one giant prompt.
6. **Markdown generation** gives us a readable final document.
7. **Streaming** improves the user experience by showing output as it arrives.

### The mental model to remember

```text
Scraper = collects information

LLM #1 = decides what information matters

Python = connects the steps together

LLM #2 = turns the information into useful content

Markdown = presents the final result
```

## 12. Review questions

Before considering the project understood, make sure you can answer these without looking at the code:

1. Why do we scrape the website before calling the brochure-writing model?
2. Why do we need `select_relevant_links()`?
3. Why is the first LLM response JSON instead of normal prose?
4. What does `json.loads(result)` do?
5. What is the difference between the system prompt and user prompt here?
6. Why do we call the LLM more than once?
7. What does `user_prompt[:5_000]` achieve?
8. What changes when `stream=True` is used?
9. What does `chunk.choices[0].delta.content` represent in the streaming loop?
10. Why is it useful to keep the API key in `.env` instead of the notebook?

**Don't just memorize the answers. Be able to trace the data from URL → scraper → LLM → Python → LLM → brochure.**